In [ ]:
# Cell 1: Imports and Setup
import pathlib
from pathlib import Path
import sys 
import os

import numpy as np
import pandas as pd

# Add the Research folder to path (parent of HarrisLabPlotting)
# test_files -> HarrisLabPlotting -> Research
sys.path.insert(0, str(Path.cwd().parent.parent))

# Import the loader
from HarrisLabPlotting.loaders import NetNeurotoolsModularityLoader

print("Imports complete!")
print(f"Working directory: {Path.cwd()}")
print(f"Added to path: {Path.cwd().parent.parent}")

In [ ]:
# Cell 2: Configuration - UPDATE THESE PATHS TO YOUR DATA LOCATION
# ================================================================

# K value to process (k=5 means 5 clusters)
K_VALUE = 5

# Path to the connectivity matrices (.npy file with shape [n_states, n_rois, n_rois])
MATRIX_PATH = Path(r"G:\My Drive\gmvae_stim_experiments_v2\experiment_k5_15_combined_20251114_163203\omst_filter_matrices\stim_matrices\k5_strategy_A_positive.npy")

# Path to netneurotools modularity results directory
NETNEUROTOOLS_DIR = Path(r"G:\My Drive\gmvae_stim_experiments_v2\experiment_k5_15_combined_20251114_163203\modularity_significance_analysis_omst_pos")

# Output directory (inside test_files)
OUTPUT_DIR = Path.cwd()  # This should be the test_files directory

# State mapping for k=5 (maps state indices to display labels)
STATE_MAPPING = {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}

print(f"K value: {K_VALUE}")
print(f"Matrix path: {MATRIX_PATH}")
print(f"NetNeurotools dir: {NETNEUROTOOLS_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\nMatrix exists: {MATRIX_PATH.exists()}")
print(f"NetNeurotools dir exists: {NETNEUROTOOLS_DIR.exists()}")

In [ ]:
# Cell 3: Load the connectivity matrices
# ========================================

# Load matrices from .npy file
# Shape should be (n_states, n_rois, n_rois) e.g., (5, 114, 114) for k=5
matrices = np.load(MATRIX_PATH)

print(f"Loaded matrices shape: {matrices.shape}")
print(f"Number of states: {matrices.shape[0]}")
print(f"Number of ROIs: {matrices.shape[1]}")
print(f"\nMatrix dtype: {matrices.dtype}")
print(f"Matrix min: {matrices.min():.4f}")
print(f"Matrix max: {matrices.max():.4f}")

In [ ]:
# Cell 4: Load modularity results using NetNeurotoolsModularityLoader
# =====================================================================

# Initialize the loader
loader = NetNeurotoolsModularityLoader(NETNEUROTOOLS_DIR)

# Load comprehensive results for k=5
k_results = loader.load_comprehensive_results(K_VALUE)

print(f"\nLoaded results for {len(k_results['states'])} states")
print("\nState summary:")
for state_data in k_results['states']:
    state_idx = state_data['state_idx']
    q_total = state_data.get('Q_total', 0)
    q_z = state_data.get('Q_z_score', 0)
    n_modules = len(np.unique(state_data['consensus']))
    print(f"  State {state_idx}: Q={q_total:.3f}, Z={q_z:.2f}, Modules={n_modules}")

In [ ]:
# Cell 4b: VERIFICATION - Check data integrity
# =============================================
# This cell verifies that the loaded data is consistent

print("="*80)
print("DATA INTEGRITY VERIFICATION")
print("="*80)

issues_found = False

for state_data in k_results['states']:
    state_idx = state_data['state_idx']
    print(f"\n--- State {state_idx} ---")
    
    # Check 1: Verify all arrays have same length
    consensus = state_data['consensus']
    if len(consensus.shape) > 1:
        consensus = consensus.flatten()
    pc = state_data['participation_coef']
    z = state_data['within_module_zscore']
    modules_df = state_data['modules_df']
    
    print(f"  Module assignments length: {len(consensus)}")
    print(f"  PC scores length: {len(pc)}")
    print(f"  Z-scores length: {len(z)}")
    print(f"  Metrics DataFrame rows: {len(modules_df)}")
    
    if not (len(consensus) == len(pc) == len(z) == len(modules_df)):
        print(f"  WARNING: Length mismatch!")
        issues_found = True
    
    # Check 2: Verify module assignments match between NPZ and CSV
    if 'module' in modules_df.columns:
        csv_modules = modules_df['module'].values
        if not np.array_equal(consensus, csv_modules):
            print(f"  WARNING: Module assignments differ between NPZ and CSV!")
            mismatches = np.sum(consensus != csv_modules)
            print(f"      Number of mismatches: {mismatches} / {len(consensus)}")
            issues_found = True
        else:
            print(f"  [OK] Module assignments match between NPZ and CSV")
    
    # Check 3: Verify expected number of ROIs (should be 114 for your atlas)
    expected_rois = matrices.shape[1]  # Get from connectivity matrix
    if len(consensus) != expected_rois:
        print(f"  WARNING: Expected {expected_rois} ROIs but got {len(consensus)}")
        issues_found = True
    else:
        print(f"  [OK] ROI count matches connectivity matrix ({expected_rois})")
    
    # Check 4: Verify PC and z-scores are not all zeros
    if np.all(pc == 0):
        print(f"  WARNING: All PC scores are zero (possible loading failure)")
        issues_found = True
    else:
        print(f"  [OK] PC scores loaded (range: {pc.min():.4f} to {pc.max():.4f})")
    
    if np.all(z == 0):
        print(f"  WARNING: All z-scores are zero (possible loading failure)")
        issues_found = True
    else:
        print(f"  [OK] Z-scores loaded (range: {z.min():.4f} to {z.max():.4f})")
    
    # Check 5: Verify no NaN values
    if np.any(np.isnan(pc)):
        print(f"  WARNING: PC scores contain NaN values")
        issues_found = True
    if np.any(np.isnan(z)):
        print(f"  WARNING: Z-scores contain NaN values")
        issues_found = True
    
    # Check 6: Show DataFrame columns for reference
    print(f"  DataFrame columns: {list(modules_df.columns)}")

print("\n" + "="*80)
if issues_found:
    print("WARNING: ISSUES FOUND - Please review warnings above")
else:
    print("[OK] ALL CHECKS PASSED - Data appears to be loaded correctly")
print("="*80)

In [ ]:
# Cell 5: Create output folder structure
# ========================================

# Create base output directory for k=5
k_output_dir = OUTPUT_DIR / f"k{K_VALUE}_data"
k_output_dir.mkdir(exist_ok=True)

# Create state folders
state_folders = {}
for state_data in k_results['states']:
    state_idx = state_data['state_idx']
    state_label = STATE_MAPPING.get(state_idx, state_idx)
    state_folder = k_output_dir / f"state_{state_label}"
    state_folder.mkdir(exist_ok=True)
    state_folders[state_idx] = state_folder
    print(f"Created folder: {state_folder}")

print(f"\nOutput structure created at: {k_output_dir}")

In [ ]:
# Cell 6: Save connectivity matrices for each state
# ===================================================

print("Saving connectivity matrices...")
print("="*60)

for state_data in k_results['states']:
    state_idx = state_data['state_idx']
    state_label = STATE_MAPPING.get(state_idx, state_idx)
    state_folder = state_folders[state_idx]
    
    # Get the connectivity matrix for this state
    connectivity_matrix = matrices[state_idx].copy()
    
    # Set negative values to 0 (as done in the pipeline)
    connectivity_matrix[connectivity_matrix < 0] = 0
    
    print(f"\nState {state_label} (index {state_idx}):")
    print(f"  Matrix shape: {connectivity_matrix.shape}")
    print(f"  Non-zero edges: {np.sum(connectivity_matrix != 0) // 2}")
    
    # Save as NumPy .npy file
    npy_path = state_folder / f"connectivity_matrix.npy"
    np.save(npy_path, connectivity_matrix)
    print(f"  Saved: {npy_path.name}")
    
    # Save as CSV file
    csv_path = state_folder / f"connectivity_matrix.csv"
    pd.DataFrame(connectivity_matrix).to_csv(csv_path, index=False, header=False)
    print(f"  Saved: {csv_path.name}")
    
    # Save as Pandas DataFrame (pickle format)
    pd_path = state_folder / f"connectivity_matrix.pkl"
    pd.DataFrame(connectivity_matrix).to_pickle(pd_path)
    print(f"  Saved: {pd_path.name}")

print("\n" + "="*60)
print("All connectivity matrices saved!")

In [ ]:
# Cell 7: Save modularity assignments for each state
# ====================================================

print("Saving modularity assignments...")
print("="*60)

for state_data in k_results['states']:
    state_idx = state_data['state_idx']
    state_label = STATE_MAPPING.get(state_idx, state_idx)
    state_folder = state_folders[state_idx]
    
    # Get module assignments (consensus clustering)
    module_assignments = state_data['consensus']
    if len(module_assignments.shape) > 1:
        module_assignments = module_assignments.flatten()
    
    print(f"\nState {state_label} (index {state_idx}):")
    print(f"  Number of nodes: {len(module_assignments)}")
    print(f"  Unique modules: {np.unique(module_assignments)}")
    
    # Save as NumPy .npy file
    npy_path = state_folder / f"module_assignments.npy"
    np.save(npy_path, module_assignments)
    print(f"  Saved: {npy_path.name}")
    
    # Save as CSV file (using 0-based roi_index to match loader expectations)
    csv_path = state_folder / f"module_assignments.csv"
    df = pd.DataFrame({
        'roi_index': np.arange(len(module_assignments)),  # 0-based indexing
        'module': module_assignments
    })
    df.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path.name}")
    
    # Save as Pandas DataFrame (pickle format)
    pd_path = state_folder / f"module_assignments.pkl"
    df.to_pickle(pd_path)
    print(f"  Saved: {pd_path.name}")

print("\n" + "="*60)
print("All modularity assignments saved!")

In [ ]:
# Cell 8: Save Participation Coefficient (PC) scores for each state
# ===================================================================

print("Saving Participation Coefficient (PC) scores...")
print("="*60)

for state_data in k_results['states']:
    state_idx = state_data['state_idx']
    state_label = STATE_MAPPING.get(state_idx, state_idx)
    state_folder = state_folders[state_idx]
    
    # Get PC scores
    pc_scores = state_data['participation_coef']
    
    print(f"\nState {state_label} (index {state_idx}):")
    print(f"  Number of nodes: {len(pc_scores)}")
    print(f"  PC range: [{pc_scores.min():.4f}, {pc_scores.max():.4f}]")
    print(f"  PC mean: {pc_scores.mean():.4f}")
    
    # Save as NumPy .npy file
    npy_path = state_folder / f"participation_coefficient.npy"
    np.save(npy_path, pc_scores)
    print(f"  Saved: {npy_path.name}")
    
    # Save as CSV file (using 0-based roi_index to match loader expectations)
    csv_path = state_folder / f"participation_coefficient.csv"
    df = pd.DataFrame({
        'roi_index': np.arange(len(pc_scores)),  # 0-based indexing
        'participation_coef': pc_scores
    })
    df.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path.name}")
    
    # Save as Pandas DataFrame (pickle format)
    pd_path = state_folder / f"participation_coefficient.pkl"
    df.to_pickle(pd_path)
    print(f"  Saved: {pd_path.name}")

print("\n" + "="*60)
print("All PC scores saved!")

In [ ]:
# Cell 9: Save Within-Module Z-scores for each state
# ====================================================

print("Saving Within-Module Z-scores...")
print("="*60)

for state_data in k_results['states']:
    state_idx = state_data['state_idx']
    state_label = STATE_MAPPING.get(state_idx, state_idx)
    state_folder = state_folders[state_idx]
    
    # Get within-module z-scores
    z_scores = state_data['within_module_zscore']
    
    print(f"\nState {state_label} (index {state_idx}):")
    print(f"  Number of nodes: {len(z_scores)}")
    print(f"  Z-score range: [{z_scores.min():.4f}, {z_scores.max():.4f}]")
    print(f"  Z-score mean: {z_scores.mean():.4f}")
    
    # Save as NumPy .npy file
    npy_path = state_folder / f"within_module_zscore.npy"
    np.save(npy_path, z_scores)
    print(f"  Saved: {npy_path.name}")
    
    # Save as CSV file (using 0-based roi_index to match loader expectations)
    csv_path = state_folder / f"within_module_zscore.csv"
    df = pd.DataFrame({
        'roi_index': np.arange(len(z_scores)),  # 0-based indexing
        'within_module_zscore': z_scores
    })
    df.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path.name}")
    
    # Save as Pandas DataFrame (pickle format)
    pd_path = state_folder / f"within_module_zscore.pkl"
    df.to_pickle(pd_path)
    print(f"  Saved: {pd_path.name}")

print("\n" + "="*60)
print("All within-module z-scores saved!")

In [ ]:
# Cell 10: Save combined metrics DataFrame for each state
# =========================================================
# This combines all modularity metrics into a single convenient file

print("Saving combined metrics DataFrames...")
print("="*60)

for state_data in k_results['states']:
    state_idx = state_data['state_idx']
    state_label = STATE_MAPPING.get(state_idx, state_idx)
    state_folder = state_folders[state_idx]
    
    # Get all metrics
    module_assignments = state_data['consensus']
    if len(module_assignments.shape) > 1:
        module_assignments = module_assignments.flatten()
    pc_scores = state_data['participation_coef']
    z_scores = state_data['within_module_zscore']
    
    # Check if we have the full metrics DataFrame from the loader
    if 'modules_df' in state_data and state_data['modules_df'] is not None:
        # Use the existing DataFrame (includes ROI names)
        combined_df = state_data['modules_df'].copy()
        # Ensure roi_index is 0-based
        if 'roi_index' in combined_df.columns:
            # Check if it's 1-based and convert to 0-based
            if combined_df['roi_index'].min() == 1:
                combined_df['roi_index'] = combined_df['roi_index'] - 1
        else:
            # Add 0-based roi_index column at the beginning
            combined_df.insert(0, 'roi_index', np.arange(len(combined_df)))
        print(f"\nState {state_label}: Using existing metrics DataFrame with {len(combined_df)} rows")
    else:
        # Create a new combined DataFrame with 0-based indexing
        combined_df = pd.DataFrame({
            'roi_index': np.arange(len(module_assignments)),  # 0-based indexing
            'module': module_assignments,
            'participation_coef': pc_scores,
            'within_module_zscore': z_scores
        })
        print(f"\nState {state_label}: Created new metrics DataFrame with {len(combined_df)} rows")
    
    print(f"  Columns: {list(combined_df.columns)}")
    print(f"  roi_index range: [{combined_df['roi_index'].min()}, {combined_df['roi_index'].max()}]")
    
    # Save as CSV file
    csv_path = state_folder / f"combined_metrics.csv"
    combined_df.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path.name}")
    
    # Save as Pandas DataFrame (pickle format)
    pd_path = state_folder / f"combined_metrics.pkl"
    combined_df.to_pickle(pd_path)
    print(f"  Saved: {pd_path.name}")

print("\n" + "="*60)
print("All combined metrics saved!")

In [ ]:
# Cell 11: Summary - List all created files
# ==========================================

print("="*80)
print("SUMMARY: All files created")
print("="*80)

for state_data in k_results['states']:
    state_idx = state_data['state_idx']
    state_label = STATE_MAPPING.get(state_idx, state_idx)
    state_folder = state_folders[state_idx]
    
    print(f"\nState {state_label} folder: {state_folder}")
    print("-"*60)
    
    # List all files in the folder
    files = sorted(state_folder.glob("*"))
    for f in files:
        file_size = f.stat().st_size / 1024  # KB
        print(f"   {f.name} ({file_size:.1f} KB)")

print("\n" + "="*80)
print("OUTPUT STRUCTURE:")
print(f"""
{k_output_dir}/
├── state_0/
│   ├── connectivity_matrix.npy    (connectivity matrix as numpy)
│   ├── connectivity_matrix.csv    (connectivity matrix as CSV)
│   ├── connectivity_matrix.pkl    (connectivity matrix as pandas pickle)
│   ├── module_assignments.npy     (module assignments as numpy)
│   ├── module_assignments.csv     (module assignments as CSV, 0-indexed)
│   ├── module_assignments.pkl     (module assignments as pandas pickle)
│   ├── participation_coefficient.npy   (PC scores as numpy)
│   ├── participation_coefficient.csv   (PC scores as CSV, 0-indexed)
│   ├── participation_coefficient.pkl   (PC scores as pandas pickle)
│   ├── within_module_zscore.npy   (z-scores as numpy)
│   ├── within_module_zscore.csv   (z-scores as CSV, 0-indexed)
│   ├── within_module_zscore.pkl   (z-scores as pandas pickle)
│   ├── combined_metrics.csv       (all metrics combined, 0-indexed)
│   └── combined_metrics.pkl       (all metrics combined)
├── state_1/
│   └── ... (same structure)
└── ... (one folder per state)

NOTE: All CSV files use 0-based roi_index (0 to 113) to match the 
connectivity matrix row/column ordering. The loader uses POSITION-BASED 
alignment, so metrics must be ordered to match the ROI coordinate table.
""")
print("="*80)
print("Done! All files saved successfully.")